# 03 — Base Model Exploration (Phi-3 Mini — Before Fine-tuning)

**Goal:** Load the base `microsoft/Phi-3-mini-4k-instruct` model in 4-bit quantization and test it on 5 medical questions.

**Day 16** of the implementation plan.

> This is your **baseline** — save these responses. After fine-tuning (Day 24), you'll compare the two side-by-side.

**⚠️ Requires GPU (RTX 4050). First run downloads ~2GB of model weights.**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Verify GPU is available before proceeding
print('PyTorch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('ERROR: No GPU detected. Check your CUDA installation.')

In [ ]:
model_name = 'microsoft/Phi-3-mini-4k-instruct'

# 4-bit quantization config — allows the 3.8B model to fit in 6GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True  # Extra memory saving
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print('Loading model (this will take a few minutes on first run)...')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
print('Model loaded!')

# Check how much VRAM is being used
used_vram = torch.cuda.memory_allocated() / 1e9
print(f'VRAM used by model: {used_vram:.2f} GB')

In [ ]:
def ask(question: str, max_new_tokens: int = 200) -> str:
    """Ask the BASE model a medical question (no fine-tuning yet)."""
    prompt = f'<|user|>\n{question}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's reply
    if '<|assistant|>' in full_response:
        return full_response.split('<|assistant|>')[-1].strip()
    return full_response.strip()

In [ ]:
# ─── Test the base model on 5 medical questions ───────────────────────────────
# IMPORTANT: Save these answers — you will compare them against your fine-tuned
# model on Day 24 to measure improvement.

questions = [
    'I have a severe headache and sensitivity to light. What could this be?',
    'What are the symptoms of type 2 diabetes?',
    'Is it safe to take ibuprofen every day?',
    'I\'ve had a persistent cough for 3 weeks. Should I see a doctor?',
    'What\'s the difference between a cold and the flu?'
]

base_responses = {}

for q in questions:
    print(f'\n{"="*60}')
    print(f'Question: {q}')
    print(f'{"="*60}')
    answer = ask(q)
    base_responses[q] = answer
    print(f'Answer: {answer}')

print('\n✓ Base model evaluation complete. Save these responses for comparison after fine-tuning.')

In [ ]:
# Save the baseline responses to a file so you can compare later
import json
import os

os.makedirs('data/processed', exist_ok=True)
with open('data/processed/baseline_responses.json', 'w') as f:
    json.dump(base_responses, f, indent=2)

print('Baseline responses saved to data/processed/baseline_responses.json')

In [ ]:
# Write in LEARNINGS.md:
# - Are the base model's answers helpful?
# - Do they sound like a real doctor?
# - Do they hallucinate (make up facts)?
# - Are they formatted consistently?